# AUDIT GATE — one run-all idempotent notebook (frozen 12-AA `9162ce44`)

**`union_bench/` is canonical.** A one-time **merge** cell copies existing Drive results (`audit_out/*`: M0 1k+10k, B1, Rprime 1k) into `union_bench/` so skip-if-done reuses them (saves ~2h GPU; no re-audit). All new outputs write straight to Drive.

- **PHASE 0** (CPU, seconds): merge · E1 read M0 per-norm · E2 per-component 12×2 + masking check · **CONCORDANCE** (per-norm survival overlap + union-AND M1a-vs-M0 → `concordance.json`; the number for the "gain routes through ℓ₁" claim).
- **PHASE 1** (GPU, gated skip-if-done): M0 (skips post-merge) · M1b · **seed-2 replicate** (gate = *done* sentinel, not ep1 snapshot) · MSD/MAX/AVG@10k optional · EXTENSION disabled.

> **Seed-2** (`REP_SEED=2`, locked). Upload each finished replicate as a folder on Drive: `seed2/M1a/{val_best.pt, train.json}` and `seed2/M0/{val_best.pt, train.json}`. The gate reads `train.json` (written only after the full 80-ep loop) — `val_best.pt` alone (an ep1 snapshot) does **not** open the gate.
>
> **Seed-2 runs 10k ONLY** (`SEED_SCALES=['10k']`). The 1k was already audited locally — **M1a_s2 0.4340 · M0_s2 0.4110 · Δ +0.0229 · LCB95 +0.0060 → SIGNIF** — and those files are not on Drive, so skip-if-done can't see them; re-running 1k would burn ~50 min of GPU for numbers we have. The 10k is the decision number.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')

## Config

In [ ]:
DRIVE       = '/content/drive/MyDrive/attackdro'
RESULTS_DIR = f'{DRIVE}/union_bench'                 # CANONICAL. all eval.json + masks land here
AUDIT_OUT   = f'{DRIVE}/audit_out'                   # legacy Drive results -> merged in once
REPO        = '/content/attackdro'
DRIVE_CIFAR_TARGZ = f'{DRIVE}/cifar-10-python.tar.gz'
DRIVE_CODE_ZIP    = f'{DRIVE}/attackdro_code.zip'
REP_SEED = 2   # locked. replicate pair on the 5070 Ti (--seed 2).
# seed-2 @1k is ALREADY audited locally (M1a_s2 0.4340 / M0_s2 0.4110 -> paired +0.0229 LCB +0.0060 SIGNIF),
# and those files are NOT on Drive, so skip-if-done cannot see them -> audit 10k only (saves ~50 min GPU).
# Set to ['1k','10k'] ONLY if you also upload the local 1k eval.json+masks into union_bench/{M1a_seed2,M0_seed2}/.
SEED_SCALES = ['10k']
import os; os.makedirs(RESULTS_DIR, exist_ok=True)

# ckpts on Drive (edit to your upload paths). seed-2 rows expect a folder with val_best.pt + train.json.
CKPT = {
  'M0':                  f'{DRIVE}/C5_M0_out/ckpt/val_best.pt',
  'M1b':                 f'{DRIVE}/C5_M1b_out/ckpt/val_best.pt',
  f'M1a_seed{REP_SEED}': f'{DRIVE}/seed{REP_SEED}/M1a/val_best.pt',
  f'M0_seed{REP_SEED}':  f'{DRIVE}/seed{REP_SEED}/M0/val_best.pt',
  'MSD':                 f'{DRIVE}/baselines/MSD.pt',
  'MAX':                 f'{DRIVE}/baselines/MAX.pt',
  'AVG':                 f'{DRIVE}/baselines/AVG.pt',
  'Rprime':              f'{DRIVE}/Rprime_out/val_best.pth',
  'B1':                  f'{DRIVE}/Bet1_out/B1_pullpush_seed0/val_best.pth',
  'B2':                  f'{DRIVE}/Bet2_out/B2_supcon_seed0/val_best.pth',
}
print('RESULTS_DIR =', RESULTS_DIR, '| REP_SEED =', REP_SEED)

## Setup — idempotent (unzip code · CIFAR tarball · RAMP symlink · autoattack)

In [ ]:
import hashlib, tarfile, zipfile, os, subprocess as _sp, importlib.util
os.makedirs(REPO, exist_ok=True)
with zipfile.ZipFile(DRIVE_CODE_ZIP) as z: z.extractall(REPO)
h=hashlib.sha256(open(DRIVE_CIFAR_TARGZ,'rb').read()).hexdigest(); assert h.startswith('6d958be074577803'),'CIFAR sha mismatch'
os.makedirs(f'{REPO}/data', exist_ok=True)
if not os.path.exists(f'{REPO}/data/cifar-10-batches-py/test_batch'):
    with tarfile.open(DRIVE_CIFAR_TARGZ) as t: t.extractall(f'{REPO}/data')
RAMP_DST=f'{REPO}/external/RAMP'; os.makedirs(f'{REPO}/external', exist_ok=True)
if os.path.exists('/content/RAMP/model_zoo/fast_models.py'):
    _sp.run(['rm','-rf',RAMP_DST]); os.symlink('/content/RAMP', RAMP_DST); print('linked /content/RAMP -> external/RAMP')
elif not os.path.exists(f'{RAMP_DST}/model_zoo/fast_models.py'):
    print('cloning RAMP...'); _sp.run(['git','clone','--depth','1','https://github.com/uiuc-focal-lab/RAMP',RAMP_DST])
if importlib.util.find_spec('autoattack') is None:
    _sp.run(['pip','install','-q','git+https://github.com/fra31/auto-attack.git'])
assert os.path.exists(f'{REPO}/scripts/eval_multinorm_audit.py'), 'harness missing — re-upload attackdro_code.zip'
assert importlib.util.find_spec('autoattack') is not None, 'autoattack not importable'
print('setup OK')

## PHASE 0 — merge legacy Drive results into canonical `union_bench/` (one-time, no-clobber)

In [ ]:
import subprocess, os
os.makedirs(RESULTS_DIR, exist_ok=True)
if os.path.isdir(AUDIT_OUT):
    subprocess.run(f'cp -rn "{AUDIT_OUT}"/* "{RESULTS_DIR}"/ 2>/dev/null', shell=True)   # -n = never overwrite
    print('merged audit_out/* -> union_bench/ (no-clobber)')
else:
    print('no audit_out/ to merge (skip)')
print('union_bench arms:', sorted(os.listdir(RESULTS_DIR)) if os.path.isdir(RESULTS_DIR) else [])

### E1 — M0 per-norm (Tab.1 / §5.2). M0 already audited (merged); NOT re-audited.

In [ ]:
import json, os
def _show_M0(scale):
    p=f'{RESULTS_DIR}/M0/'+('' if scale=='1k' else '10k/')+'eval.json'
    if not os.path.exists(p): print(f'M0 {scale}: eval.json not found ({p})'); return
    d=json.load(open(p)); pn=d.get('per_norm_audit',{})
    print(f"M0 {scale:3s}:  clean={d['clean_acc']:.4f}  union(12-AA)={d['full_audit_union']:.4f}")
    if pn:
        print(f"   per-norm audit_acc:  linf={pn['audit_acc_linf']:.4f}  l2={pn['audit_acc_l2']:.4f}  l1={pn['audit_acc_l1']:.4f}")
        print(f"   black_box_gap     :  linf={pn['black_box_gap_linf']:+.4f} l2={pn['black_box_gap_l2']:+.4f} l1={pn['black_box_gap_l1']:+.4f}  (neg => no masking)")
_show_M0('1k'); _show_M0('10k')

### E2 — copy M1a (zip → union_bench) · per-component 12×2 · masking check → m1a_vs_m0.json

In [ ]:
import json, os, shutil
for scale in ['1k','10k']:                                   # M1a masks/eval ship in the code zip
    for fn in ['eval.json','masks_multinorm_v1.npz']:
        src=f'{REPO}/results/eval/union_bench/M1a/'+('' if scale=='1k' else '10k/')+fn
        dst=f'{RESULTS_DIR}/M1a/'+('' if scale=='1k' else '10k/')+fn
        if os.path.exists(src) and not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy(src, dst); print('copied ->', dst)
rep={'scale':'10k'}
try:
    dm1=json.load(open(f'{RESULTS_DIR}/M1a/10k/eval.json'))['per_attack']
    dm0=json.load(open(f'{RESULTS_DIR}/M0/10k/eval.json'))['per_attack']
    print('per-component robust_acc (12x2):'); print(f"  {'attack':16s} {'M1a':>7s} {'M0':>7s}")
    tab={}
    for k in sorted(dm1):
        r1=dm1[k]['robust_acc']; r0=dm0.get(k,{}).get('robust_acc'); tab[k]={'M1a':r1,'M0':r0}
        print(f"  {k:16s} {r1:7.4f} {('%7.4f'%r0) if isinstance(r0,(int,float)) else '   n/a'}")
    rep['per_component']=tab
    print('\nmasking check (square vs worst white-box, per norm):')
    for nm in ['linf','l2','l1']:
        for lab,dd in [('M1a',dm1),('M0',dm0)]:
            wb=min(dd[f'apgd_ce_{nm}']['robust_acc'],dd[f'apgd_dlr_{nm}']['robust_acc'],dd[f'fab_t_{nm}']['robust_acc'])
            sq=dd[f'square_{nm}']['robust_acc']
            print(f"  {lab} {nm:4s}: worst_wb={wb:.4f} square={sq:.4f} -> {'NO-MASK (wb stronger)' if sq>=wb else 'MASK? square<wb'}")
    os.makedirs(f'{RESULTS_DIR}/_phase0', exist_ok=True)
    json.dump(rep, open(f'{RESULTS_DIR}/_phase0/m1a_vs_m0.json','w'), indent=1)
    print('\nsaved ->', f'{RESULTS_DIR}/_phase0/m1a_vs_m0.json')
except Exception as e: print('E2 skipped (need M0 merged):', e)

### CONCORDANCE — per-norm survival overlap + union-AND (M1a vs M0) → concordance.json
The number behind *"the gain routes through ℓ₁ / where do the two models disagree"*.

In [ ]:
import json, os, numpy as np
def _masks(name):
    p=f'{RESULTS_DIR}/{name}/10k/masks_multinorm_v1.npz'
    if not os.path.exists(p): raise FileNotFoundError(p)
    d=np.load(p); return {k:d[k].astype(bool) for k in d.files if k!='metadata_json'}
def _pernorm(m,norm):
    ks=[k for k in m if k.endswith(norm)]; u=np.ones(len(next(iter(m.values()))),bool)
    for k in ks: u=u&m[k]
    return u
def _ov(a,b):
    return {'both':int((a&b).sum()),'A_only':int((a&~b).sum()),'B_only':int((~a&b).sum()),
            'neither':int((~a&~b).sum()),'A_acc':float(a.mean()),'B_acc':float(b.mean()),
            'net_A_minus_B':float(a.mean()-b.mean())}
try:
    M1a,M0=_masks('M1a'),_masks('M0'); n=len(next(iter(M1a.values())))
    conc={'scale':'10k','n':int(n),'A':'M1a','B':'M0','per_norm':{},'union':{}}
    print('CONCORDANCE M1a-vs-M0 (10k) — per-norm survival overlap (per-example AND over that norm 4 attacks):')
    print(f"  {'norm':5s} {'both':>6s} {'M1a_only':>9s} {'M0_only':>8s} {'neither':>8s}   {'M1a':>7s} {'M0':>7s} {'net':>8s}")
    for nm in ['linf','l2','l1']:
        ov=_ov(_pernorm(M1a,nm),_pernorm(M0,nm)); conc['per_norm'][nm]=ov
        print(f"  {nm:5s} {ov['both']:6d} {ov['A_only']:9d} {ov['B_only']:8d} {ov['neither']:8d}   {ov['A_acc']:7.4f} {ov['B_acc']:7.4f} {ov['net_A_minus_B']:+8.4f}")
    ua=np.ones(n,bool); [ua.__iand__(v) for v in M1a.values()]
    ub=np.ones(n,bool); [ub.__iand__(v) for v in M0.values()]
    conc['union']=_ov(ua,ub)
    print(f"\n  UNION(12): M1a={ua.mean():.4f} M0={ub.mean():.4f} net={ua.mean()-ub.mean():+.4f}  (M1a-only {conc['union']['A_only']} / M0-only {conc['union']['B_only']})")
    gain=max(conc['per_norm'].items(), key=lambda kv: kv[1]['net_A_minus_B'])
    print(f"  => per-norm gain concentrates in: {gain[0]} (net {gain[1]['net_A_minus_B']:+.4f})")
    os.makedirs(f'{RESULTS_DIR}/_phase0', exist_ok=True)
    json.dump(conc, open(f'{RESULTS_DIR}/_phase0/concordance.json','w'), indent=1)
    print('  saved ->', f'{RESULTS_DIR}/_phase0/concordance.json')
except Exception as e: print('concordance skipped (need M0 + M1a masks):', e)

## PHASE 1 — GPU queue helpers (audit → Drive, union, paired, done-gate)

In [ ]:
import subprocess, numpy as np, json, os
C1K ='configs/eval/audit_cifar10_preactrn18_multinorm_v3A_testfinal.yaml'
C10K='results/eval/union_bench/_config/audit_v3A_test_10k.yaml'
def _p(name,scale,fn): return f'{RESULTS_DIR}/{name}/'+('' if scale=='1k' else f'{scale}/')+fn
def _completion_ok(ckpt, min_epoch=70):
    "done-sentinel gate: c5 train.json (written only after the full loop) or ramp val_best_meta with epoch>=min."
    d=os.path.dirname(ckpt)
    for tj in [os.path.join(d,'train.json'), ckpt.rsplit('.',1)[0]+'_train.json']:
        if os.path.exists(tj):
            try:
                t=json.load(open(tj)); hist=t.get('history',[])
                ep=max((h.get('epoch',-1) for h in hist), default=t.get('best_epoch',-1))
                return (ep>=min_epoch), f'train.json epoch {ep}'
            except Exception as e: return False, f'train.json unreadable ({e})'
    for m in [os.path.join(d,'val_best_meta.pth'), ckpt.rsplit('.',1)[0]+'_meta.pth']:
        if os.path.exists(m):
            try:
                import torch; ep=torch.load(m,map_location='cpu',weights_only=False).get('epoch',-1)
                return (ep>=min_epoch), f'meta epoch {ep}'
            except Exception: pass
    return False, 'no done-sentinel (upload train.json or val_best_meta with epoch>=70 next to ckpt)'
def audit_one(ckpt, family, name, scale):
    out=_p(name,scale,'eval.json')
    if os.path.exists(out):
        u=json.load(open(out)).get('full_audit_union'); print(f'  SKIP {name} {scale} (done union={u:.4f})'); return u
    if not ckpt or not os.path.exists(ckpt):
        print(f'  GATE {name} {scale}: ckpt absent -> skip ({ckpt})'); return None
    os.makedirs(os.path.dirname(out), exist_ok=True)
    cfg=C1K if scale=='1k' else C10K
    if family in ('robustdro','ramp'):
        cmd=['python','scripts/eval_multinorm_audit.py','--config',cfg,'--checkpoint',ckpt,'--model-family',family,
             '--run-id',f'{name}_{scale}','--checkpoint-role','val_best','--out',out,'--export-masks','--bs','128']
    else:
        cmd=['python','scripts/dev/union_bench_eval.py','--config',cfg,'--checkpoint',ckpt,'--arch',family,
             '--run-id',f'{name}_{scale}','--checkpoint-role','released','--out',out,'--export-masks','--bs','128']
    print(f'  AUDIT {name} {scale} ...'); r=subprocess.run(cmd,cwd=REPO,capture_output=True,text=True)
    if not os.path.exists(out):
        print('  --- STDOUT ---'); print(r.stdout[-1200:]); print('  --- STDERR ---'); print(r.stderr[-1800:])
        print(f'  !! FAILED {name} {scale}'); return None
    u=json.load(open(out))['full_audit_union']; print(f'    {name} {scale} union={u:.4f}  -> {out}'); return u
def _union(name,scale):
    fn='masks_multinorm_v1.npz'
    cands=[_p(name,scale,fn), f'{REPO}/results/eval/union_bench/{name}/'+('' if scale=='1k' else f'{scale}/')+fn]
    p=next((c for c in cands if os.path.exists(c)), None)
    if p is None: raise FileNotFoundError(f'{name} {scale} masks missing')
    d=np.load(p); ns=[k for k in d.files if k!='metadata_json']; u=np.ones(len(d[ns[0]]),bool)
    [u.__iand__(d[n].astype(bool)) for n in ns]; return u
def paired(a,b,scales=('1k','10k')):
    res={}
    for s in scales:
        try: ua,ub=_union(a,s),_union(b,s)
        except Exception as e: print(f'  {s}: {a}/{b} masks missing ({e})'); continue
        rng=np.random.default_rng(0); n=len(ua); D=[]
        for _ in range(10000): i=rng.integers(0,n,n); D.append(ua[i].mean()-ub[i].mean())
        lcb=float(np.quantile(D,0.05)); delta=float(np.mean(D)); res[s]=(float(ua.mean()),float(ub.mean()),delta,lcb)
        print(f'  {s}: {a}={ua.mean():.4f} {b}={ub.mean():.4f}  {a}-{b}={delta:+.4f} LCB95={lcb:+.4f} -> {"SIGNIF" if lcb>0 else "CI incl 0"}')
    return res
print('helpers ready — outputs ->', RESULTS_DIR)

### FAST-REVIEW audit path (standing rule for every arm)

Before committing a GPU to the full 12-AA @10k of any arm, do a cheap look first. Three steps
with a decision gate between them:

1. **PROBE** — `apgd_ce_linf` only, on the 10k set (one attack, a few minutes). Compare its
   `robust_acc` immediately to the same component of the comparison arm.
2. **1k full 12-AA** — the whole audit on the 1k subset → an approximate union (historical
   1k→10k drift is about **−2pp**, so subtract ~0.02 to estimate the 10k union).
3. **STOP for decision** — do not auto-promote. A human decides which arm graduates to the
   full 12-AA @10k.

`fast_review(name, ckpt, family, baseline=...)` runs steps 1–2 and stops. To promote, call
`run_audit(...)` (defined below in the queue helpers) or enable the arm in the QUEUE.

In [ ]:
import re
def _comp10k(name, attack):
    "read a single component's robust_acc from an already-audited arm's 10k eval.json"
    p=_p(name,'10k','eval.json')
    if not os.path.exists(p): return None
    return (json.load(open(p)).get('per_attack',{}).get(attack,{}) or {}).get('robust_acc')
def _union10k(name):
    p=_p(name,'10k','eval.json')
    return json.load(open(p)).get('full_audit_union') if os.path.exists(p) else None
def probe_linf(name, ckpt, family):
    "step 1: apgd_ce_linf @10k via the frozen harness (reuses run_attack_mask; writes nothing)"
    env=dict(os.environ, ATTACKDRO_ROOT=REPO)
    cmd=['python','scripts/dev/probe_attack.py','--config',C10K,'--checkpoint',ckpt,
         '--model-family',family,'--attack','apgd_ce_linf']
    r=subprocess.run(cmd,cwd=REPO,capture_output=True,text=True,env=env)
    line=next((l for l in r.stdout.splitlines() if l.startswith('PROBE')), None)
    if line is None:
        print(r.stdout[-1000:]); print('STDERR:',r.stderr[-1500:]); raise RuntimeError('probe failed')
    print('  ', line)
    m=re.search(r'robust_acc=([0-9.]+)', line); return float(m.group(1))
def fast_review(name, ckpt, family, baseline=None):
    assert os.path.exists(ckpt), f'ckpt not found: {ckpt}'
    print(f'=== FAST REVIEW: {name} (baseline={baseline}) ===')
    print('[1] PROBE apgd_ce_linf @10k ...')
    linf=probe_linf(name, ckpt, family)
    if baseline is not None:
        b=_comp10k(baseline,'apgd_ce_linf')
        if b is not None: print(f'    Δ_linf  {name}-{baseline} = {linf:.4f} - {b:.4f} = {linf-b:+.4f}')
        else: print(f'    ({baseline} apgd_ce_linf @10k not on Drive to compare)')
    print('[2] 1k full 12-AA ...')
    u1k=audit_one(ckpt, family, name, '1k')     # writes union_bench/<name>/eval.json + masks
    if u1k is not None:
        est=u1k-0.02
        print(f'    {name} 1k union = {u1k:.4f}  -> est 10k ~= {est:.4f} (historical 1k->10k drift ~ -2pp)')
        if baseline is not None:
            bu=_union10k(baseline)
            if bu is not None: print(f'    vs {baseline} 10k union {bu:.4f}: est Δ ~ {est-bu:+.4f}')
    print('[3] STOP — decision needed: run full 12-AA @10k for this arm?')
    print(f'    to promote:  audit_one("{ckpt}","{family}","{name}","10k")   # full 12-AA @10k + masks')
    print('    (or add it to the QUEUE with scales=["10k"] and re-run PHASE 1)')
print('fast_review ready')

### PROBE_LAST — last-epoch checkpoint as a selection-robustness DIAGNOSTIC

**Purpose:** check whether the +2.87pp finding depends on the checkpoint-selection rule.
`val_best` stays the headline (conservative). If Δ holds under *both* `val_best` and
`last-epoch`, the paper is stronger (an appendix sentence). If it moves, we know immediately
it is selection-sensitive. This is a robustness check, **not** a way to pick a better number.

**Guardrails (printed on every run, and enforced by the helper):**
- Last-epoch is run for the **whole pair** (M1a + M0, same seed) — never one arm alone.
- **Never** take the higher-on-test ckpt per arm. Only Δ under a **single uniform rule** is reported.
- M0 loses ~**−3.9pp** proxy after its peak, so M0's last-epoch is *below* its val_best. Using
  last-epoch therefore **inflates** the M1a−M0 gap artificially. `val_best` is the honest denominator.

Follows the standing 3-step shape: (i) `apgd_ce_linf` @1k on `last.pt`, printed next to the
`val_best` number; (ii) if you decide, full 12-AA @1k on `last.pt`. Last-epoch results are
written to a **`<arm>_last`** name — they never overwrite the val_best results.

> Needs `last.pt` on Drive next to each arm's `val_best.pt`. `c5_fromscratch` saves it in the
> same backbone format. M1a seed0 last.pt sha256 starts `e2b185c1` (val_best is `f02924cb`).

In [ ]:
def _probe1(ckpt, family, cfg=C1K, attack='apgd_ce_linf'):
    env=dict(os.environ, ATTACKDRO_ROOT=REPO)
    r=subprocess.run(['python','scripts/dev/probe_attack.py','--config',cfg,'--checkpoint',ckpt,
                      '--model-family',family,'--attack',attack],cwd=REPO,capture_output=True,text=True,env=env)
    line=next((l for l in r.stdout.splitlines() if l.startswith('PROBE')), None)
    if line is None: print(r.stdout[-800:]); print('STDERR:',r.stderr[-1200:]); return None
    return float(re.search(r'robust_acc=([0-9.]+)', line).group(1))

_LAST_WARN = ("PROBE_LAST — DIAGNOSTIC selection-robustness check, NOT cherry-picking.\n"
              "  * both arms of the pair, SAME rule; never pick the higher-on-test ckpt per arm.\n"
              "  * val_best is the honest headline denominator (kept).\n"
              "  * M0 drops ~-3.9pp proxy after its peak -> M0_last < M0_val_best -> last-epoch\n"
              "    INFLATES the M1a-M0 gap. Read Δ only under one uniform rule.")

def probe_last_pair(a_name, a_vb, b_name, b_vb, family='robustdro'):
    print('='*68); print(_LAST_WARN); print('='*68)
    res={}
    print(f"  {'arm':6} {'ckpt':9} apgd_ce_linf@1k")
    for name, vb in [(a_name,a_vb),(b_name,b_vb)]:
        for rule, ck in [('val_best', vb), ('last', vb.replace('val_best.pt','last.pt'))]:
            if not os.path.exists(ck):
                print(f"  {name:6} {rule:9} -- ckpt not on Drive: {ck}"); res[(name,rule)]=None; continue
            v=_probe1(ck, family); res[(name,rule)]=v
            print(f"  {name:6} {rule:9} {v:.4f}" if v is not None else f"  {name:6} {rule:9} probe FAILED")
    print('  ' + '-'*30)
    for rule in ['val_best','last']:
        va, vb2 = res.get((a_name,rule)), res.get((b_name,rule))
        if va is not None and vb2 is not None:
            print(f"  Δ_linf({rule:8}) = {a_name}-{b_name} = {va-vb2:+.4f}   ({va:.4f} - {vb2:.4f})")
    if all(res.get((n,r)) is not None for n in (a_name,b_name) for r in ('val_best','last')):
        dv=res[(a_name,'val_best')]-res[(b_name,'val_best')]; dl=res[(a_name,'last')]-res[(b_name,'last')]
        print(f"  -> Δ_linf holds under BOTH rules" if (dv>0 and dl>0) else
              f"  -> Δ_linf SELECTION-SENSITIVE (val_best {dv:+.4f} vs last {dl:+.4f})")
    print("  DECISION: promote a pair to full 12-AA @1k on last.pt? results -> <arm>_last (no overwrite):")
    print(f"    audit_one('{a_vb.replace('val_best.pt','last.pt')}','{family}','{a_name}_last','1k')")
    print(f"    audit_one('{b_vb.replace('val_best.pt','last.pt')}','{family}','{b_name}_last','1k')")
    return res
print('probe_last_pair ready')

### QUEUE manifest + gated loop  (skip-if-done · ckpt-gate · seed done-gate)

In [ ]:
QUEUE = [
 {'name':'M0','ckpt':CKPT['M0'],'family':'robustdro','scales':['1k','10k'],'enabled':True,
  'note':'skips post-merge (eval.json present) — no re-audit'},
 {'name':'M1b','ckpt':CKPT['M1b'],'family':'robustdro','scales':['1k','10k'],'enabled':True,
  'note':"adv-vs-clean neg; feeds paired('M1a','M1b') @10k"},
 {'name':f'M1a_seed{REP_SEED}','ckpt':CKPT[f'M1a_seed{REP_SEED}'],'family':'robustdro','scales':SEED_SCALES,'enabled':True,'gate':'done',
  'note':'seed-2 replicate; gate = train.json done-sentinel; 1k already done locally -> 10k only'},
 {'name':f'M0_seed{REP_SEED}','ckpt':CKPT[f'M0_seed{REP_SEED}'],'family':'robustdro','scales':SEED_SCALES,'enabled':True,'gate':'done',
  'note':'seed-2 replicate; both present -> paired'},
 {'name':'msd','ckpt':CKPT['MSD'],'family':'robust_union_preact','scales':['10k'],'enabled':True,'note':'@10k OPTIONAL (upload MSD.pt)'},
 {'name':'max','ckpt':CKPT['MAX'],'family':'robust_union_preact','scales':['10k'],'enabled':True,'note':'@10k OPTIONAL'},
 {'name':'avg','ckpt':CKPT['AVG'],'family':'robust_union_preact','scales':['10k'],'enabled':True,'note':'@10k OPTIONAL'},
 # ---------------- EXTENSION (disabled by default) ----------------
 {'name':'Rprime','ckpt':CKPT['Rprime'],'family':'ramp','scales':['10k'],'enabled':False,'note':'EXT: R′ 10k (only truly-missing) -> paired(B1,Rprime)'},
 {'name':'B2','ckpt':CKPT['B2'],'family':'ramp','scales':['1k','10k'],'enabled':False,'note':'EXT: paired(B2,Rprime)'},
]
UNIONS={}
for job in QUEUE:
    if not job['enabled']: print(f"DISABLED {job['name']:14s} — {job['note']}"); continue
    print(f"== {job['name']} ({job['family']}) — {job['note']}")
    if job.get('gate')=='done':
        ok,why=_completion_ok(job['ckpt'])
        if not ok: print(f"  GATE {job['name']}: NOT done -> skip ({why})"); continue
        print(f"  gate OK ({why})")
    for s in job['scales']:
        u=audit_one(job['ckpt'], job['family'], job['name'], s)
        if u is not None: UNIONS[(job['name'],s)] = u
print('\nunions this run:', {f'{k[0]}@{k[1]}':round(v,4) for k,v in UNIONS.items()})

### Auto-paired

In [ ]:
print('=== AUTO-PAIRED ===')
if os.path.exists(_p('M0','10k','masks_multinorm_v1.npz')):
    print("paired('M1a','M0'):"); r=paired('M1a','M0')
    if '10k' in r:
        _,m0,delta,lcb=r['10k']
        if abs(m0-0.3886)>0.002: print(f'  ⚠ M0 10k union {m0:.4f} drift >0.2pp from recorded 0.3886 (Square stochastic)')
        print('  ✓ reproduces Claim A (+2.87/LCB+2.34)' if (abs(delta-0.0287)<=0.003 and abs(lcb-0.0234)<=0.003)
              else f'  ⚠ M1a-M0 10k ({delta:+.4f}/LCB{lcb:+.4f}) != recorded (+0.0287/+0.0234)')
if os.path.exists(_p('M1b','10k','eval.json')):
    print("paired('M1a','M1b') @10k:"); paired('M1a','M1b',scales=('10k',))
sN=f'seed{REP_SEED}'; a_sN,b_sN=f'M1a_{sN}',f'M0_{sN}'
if os.path.exists(_p(a_sN,'10k','eval.json')) and os.path.exists(_p(b_sN,'10k','eval.json')):
    print(f"paired seed-{REP_SEED} ({a_sN} - {b_sN}):"); rN=paired(a_sN,b_sN,scales=tuple(SEED_SCALES))
    if '10k' in rN:
        lcbN=rN['10k'][3]
        print(f'  per-seed gaps @10k:  seed0 +0.0287 (LCB +0.0234)  |  seed{REP_SEED} {rN["10k"][2]:+.4f} (LCB {rN["10k"][3]:+.4f})')
        print(f'  seed-min LCB @10k = {min(0.0234,lcbN):+.4f}  (conservative Claim-A across replicates)')
for a in ['B1','B2']:
    if os.path.exists(_p('Rprime','10k','eval.json')) and (
        os.path.exists(_p(a,'10k','eval.json')) or os.path.exists(f'{REPO}/results/eval/union_bench/{a}/10k/masks_multinorm_v1.npz')):
        print(f"paired('{a}','Rprime'):"); paired(a,'Rprime',scales=('10k',))

## FAST-REVIEW: M0_full (matched full-budget control)

Early read on `M0_full` **before** deciding on the full 12-AA @10k. Upload
`C5_full/M0_full/val_best.pt` to Drive first (after the ~08:28 ICT run finishes).

Baseline is **M1a_full** (already audited @10k on Drive): apgd_ce_linf **0.4608**, union **0.4201**.

> **Coordination:** a local waiter is ALSO running the authoritative full-10k M0_full audit
> hands-free (~13:00 ICT) — that is the copy used for `paired('M1a_full','M0_full')`. This
> Colab cell is only the early look. If you *do* run full-10k here too, cross-check it against
> the local result (drift < 0.2pp) before using it for the paired test.

In [ ]:
M0_FULL_CKPT = f'{DRIVE}/C5_full/M0_full/val_best.pt'   # upload after training finishes
if os.path.exists(M0_FULL_CKPT):
    fast_review('M0_full', M0_FULL_CKPT, 'robustdro', baseline='M1a_full')
else:
    print('M0_full val_best.pt not on Drive yet:', M0_FULL_CKPT)
    print('Upload it after the ~08:28 ICT run finishes, then re-run this cell.')

## MSD baseline — public locuslab/robust_union ckpt @10k (reference bar)

Standalone audit of the original public **MSD** checkpoint (Maini et al. 2020). It is a
*reference bar*, not a CLAMP arm, so we **skip the probe** and run the full 12-AA @10k directly
(we already have msd @1k = 0.442). family = `robust_union_preact` (MSD's own arch), role `released`.

**Upload exactly here** (this is `CKPT['MSD']`): `MyDrive/attackdro/baselines/MSD.pt`
— i.e. `/content/drive/MyDrive/attackdro/baselines/MSD.pt`.
Source file on the PC: `external/robust_union/CIFAR10/Selected/MSD.pt` (22.4 MB).
It must be sha256 `482bf2876572cdcadb1edd6de51e173ac50de79bee0351ba5cf96c6e3c6384bc`
(the exact ckpt behind the 1k=0.442 reference). Needs code zip `679299d6…` (ROOT fix + model).

In [ ]:
import hashlib, json, os
if not os.path.exists(CKPT['MSD']):
    print('MSD.pt not on Drive. Upload the PC file external/robust_union/CIFAR10/Selected/MSD.pt')
    print('  ->', CKPT['MSD'])
else:
    h = hashlib.sha256(open(CKPT['MSD'],'rb').read()).hexdigest()
    ok = h == '482bf2876572cdcadb1edd6de51e173ac50de79bee0351ba5cf96c6e3c6384bc'
    print(f"sha256 {h[:16]}...  {'MATCH (public MSD)' if ok else '** MISMATCH -- not the reference ckpt **'}")
    assert ok, 'MSD.pt is not the expected public checkpoint; do not audit a wrong ckpt.'
    # reference bar -> full 12-AA @10k directly (skip probe); skip-if-done
    audit_one(CKPT['MSD'], 'robust_union_preact', 'msd', '10k')
    p = _p('msd','10k','eval.json')
    if os.path.exists(p):
        d = json.load(open(p)); pn = d.get('per_norm_audit',{})
        print(f"\nmsd 10k: clean {d['clean_acc']:.4f} | union {d['full_audit_union']:.4f}  (1k ref 0.442)")
        print(f"  per-norm audit_acc: linf {pn['audit_acc_linf']:.4f}  l2 {pn['audit_acc_l2']:.4f}  l1 {pn['audit_acc_l1']:.4f}")
        print(f"  black_box_gap:      linf {pn['black_box_gap_linf']:+.4f} l2 {pn['black_box_gap_l2']:+.4f} l1 {pn['black_box_gap_l1']:+.4f}  (neg = no masking)")

## PROBE_LAST: headline pair M1a / M0 (seed 0)

Selection-robustness diagnostic on the +2.87pp finding. Needs, on Drive, each arm's
`val_best.pt` **and** `last.pt` (same folder). Edit the two `val_best.pt` paths below to
where your seed-0 ckpts live; `last.pt` is found as their sibling.

M1a seed0 last.pt sha256 starts `e2b185c1` (val_best `f02924cb`) — upload it if it is not on
Drive yet. If an arm's `last.pt` is missing, that row is skipped with a message (not guessed).

In [ ]:
PAIR = {
    'M1a': f'{DRIVE}/C5_M1a_out/ckpt/val_best.pt',   # <- edit to your M1a seed0 ckpt folder
    'M0' : f'{DRIVE}/C5_M0_out/ckpt/val_best.pt',    # <- edit to your M0  seed0 ckpt folder
}
probe_last_pair('M1a', PAIR['M1a'], 'M0', PAIR['M0'], family='robustdro')

## B2 — RobustBench base reproduce-check (GATE · Track B / explore #3, run in a SPARE session)

Before fine-tuning a public ℓ∞-AT base with CLAMP, prove we load & eval it correctly. Base
**`Sehwag2021Proxy_R18`** (ResNet-18, feat 512). **Targets ±1pp: clean 84.59, AA-ℓ∞ 55.54**
(RobustBench published). Normalization is settled *empirically* here (feed [0,1]; if clean is
far off, add CIFAR mean/std). **Miss → STOP, debug, do NOT proceed to B3.** Track A first — only
run this in a spare Colab session.

In [ ]:
import subprocess, sys, os, json, hashlib, numpy as np
RB_NAME='Sehwag2021Proxy_R18'; PUB_CLEAN=0.8459; PUB_AA_LINF=0.5554
subprocess.run([sys.executable,'-m','pip','install','-q','robustbench'])          # v1.1 ok
subprocess.run([sys.executable,'-m','pip','install','-q','-U','gdown'])           # 6.x clicks the gdrive virus-scan token
import torch
CK=f'{REPO}/models/cifar10/Linf/{RB_NAME}.pt'
_orig=torch.load; torch.load=lambda *a,**k:_orig(*a,**{**k,'weights_only':False})  # torch 2.x vs RB
try:
    from robustbench.utils import load_model
    base=load_model(model_name=RB_NAME, dataset='cifar10', threat_model='Linf', model_dir=f'{REPO}/models')
finally: torch.load=_orig
sha=hashlib.sha256(open(CK,'rb').read()).hexdigest(); print('ckpt sha256:', sha[:16])
base=base.cuda().eval()

# CLEAN reproduce on the FULL 10k test set (same data as the published number -> no sampling gap;
# no attack -> cheap). This both settles normalization and gives an EXACT loading-correctness check.
sys.path.insert(0,f'{REPO}/scripts'); sys.path.insert(0,f'{REPO}/src')
import eval_multinorm_audit as H
cfg10=H.load_audit_config(H.repo_path(C10K)); x,y,_,_=H.load_subset(cfg10); xg,yg=x.cuda(),y.cuda()  # full 10k
def clean_acc(fn):
    c=0
    with torch.no_grad():
        for i in range(0,len(yg),512): c+=(fn(xg[i:i+512]).argmax(1)==yg[i:i+512]).sum().item()
    return c/len(yg)
mean=torch.tensor([0.4914,0.4822,0.4465]).view(1,3,1,1).cuda(); std=torch.tensor([0.2471,0.2435,0.2616]).view(1,3,1,1).cuda()
raw=clean_acc(lambda z: base(z)); nrm=clean_acc(lambda z: base((z-mean)/std))
use_norm = abs(nrm-PUB_CLEAN) < abs(raw-PUB_CLEAN); best=nrm if use_norm else raw
print(f'clean @10k (full test): [0,1]={raw:.4f}  normalized={nrm:.4f}  (published {PUB_CLEAN})')
print(f'-> convention: {"NORMALIZE (mean/std)" if use_norm else "[0,1]"}  | clean {best:.4f} vs pub (Δ {best-PUB_CLEAN:+.4f})')
assert abs(best-PUB_CLEAN)<=0.01, (f'clean {best:.4f} off published {PUB_CLEAN} by >1pp on the FULL test set '
                                   f'-> STOP & debug (the other convention gave {nrm if not use_norm else raw:.4f})')
print('  CLEAN reproduce PASS (exact test set, ±1pp) -> loading correct, convention settled')

# AA-Linf: fast @1k SANITY now (1k sampling -> ±2pp band); exact ±1pp needs @10k (~1.5h, note below).
out=f'{RESULTS_DIR}/rb_Sehwag_R18/eval.json'; os.makedirs(os.path.dirname(out),exist_ok=True)
cmd=[sys.executable,f'{REPO}/scripts/dev/union_bench_eval.py','--config',C1K,'--checkpoint',CK,
     '--arch','robustbench','--run-id',f'{RB_NAME}_reproduce_1k','--checkpoint-role','published',
     '--out',out,'--export-masks','--bs','128'] + (['--normalize'] if use_norm else [])
print('\nAUDIT AA-Linf @1k (fast sanity) ...')
r=subprocess.run(cmd,cwd=REPO,capture_output=True,text=True)
if not os.path.exists(out): print(r.stdout[-1500:]); print('STDERR',r.stderr[-2000:]); raise RuntimeError('reproduce audit failed')
d=json.load(open(out)); aa=d['per_norm_audit']['audit_acc_linf']
print(f"AA-Linf @1k = {aa:.4f}  (published {PUB_AA_LINF} on 10k)  Δ {aa-PUB_AA_LINF:+.4f}")
clean_ok = abs(best-PUB_CLEAN)<=0.01; aa_ok = abs(aa-PUB_AA_LINF)<=0.02   # 1k sampling band on AA
print('  GATE:', 'PASS -> proceed to B3 (finetune_colab)' if (clean_ok and aa_ok)
      else 'AA-Linf @1k outside ±2pp -> confirm @10k (--config C10K, ~1.5h) before B3')
print('  note: clean is the exact (10k) check; AA-Linf here is a 1k sanity. For the paper, run AA-Linf @10k.')
d['reproduce']={'ckpt_sha256':sha,'published_clean':PUB_CLEAN,'published_aa_linf':PUB_AA_LINF,
                'clean_10k':best,'aa_linf_1k':aa,'normalize':bool(use_norm),
                'clean_gate_pass':bool(clean_ok),'aa_gate_1k_pass':bool(aa_ok)}
json.dump(d,open(out,'w'),indent=2); print('  logged provenance ->', out)

## Summary — arm × scale × union × masks

In [ ]:
import glob, json, os
rows=[]
for ej in glob.glob(RESULTS_DIR+'/*/eval.json')+glob.glob(RESULTS_DIR+'/*/10k/eval.json'):
    rel=ej[len(RESULTS_DIR)+1:]; parts=rel.split('/'); name=parts[0]; scale='10k' if '10k' in parts else '1k'
    try: u=json.load(open(ej)).get('full_audit_union')
    except Exception: u=None
    mk=os.path.join(os.path.dirname(ej),'masks_multinorm_v1.npz')
    rows.append((name,scale,f'{u:.4f}' if isinstance(u,(int,float)) else '—','Y' if os.path.exists(mk) else '-'))
rows.sort()
print(f"{'arm':18s} {'scale':5s} {'union':8s} masks"); print('-'*42)
for n,s,u,m in rows: print(f"{n:18s} {s:5s} {u:8s} {m}")
print('\n(RESULTS_DIR =', RESULTS_DIR, ')')